# Phase 0 — Infrastructure & correctness controls

**Goal (RESEARCH_PLAN §0).** Everything downstream needs three primitives the
repo did not have:

1. a *deterministic* proposal log-probability `log q(config)` for an **arbitrary**
   configuration (the sampler only returned `log q` for a *drawn* config);
2. the exact target `log π(config) = −β·E(config) − log Z`;
3. a full-enumeration harness over all `2^{Lx·Ly}` configs giving `w = π/q` and
   the true Mengersen–Tweedie constant `C = max_x w(x)`.

These live in [`tools/tnmh_tools.jl`](tools/tnmh_tools.jl) and reuse the exact
conditional machinery of `sample_config_opt`/`sample_classical_1d` (only the
`rand()` draw is replaced by pinning to the supplied spins).

**Before trusting anything we run two controls (plan §0):**
- **A.** `Σ_x q(x) = 1` — `log q` is a genuine normalised pmf.
- **B.** at `D ≥ 2^{⌈Ly/2⌉}` the boundary MPS is lossless ⇒ the proposal is exact
  ⇒ `C = 1.0000` and acceptance `= 1`. Any deviation is a bug in the pinned
  `log q` evaluator.

> Reminder (CLAUDE.md §2a): the existing notebook tracks the **per-layer rate ρ**,
> not the full-lattice `C`. This phase builds the *first in-repo true-`C`* path.


In [1]:
using Plots, JLD2, Statistics, LinearAlgebra, DelimitedFiles, Printf
include("main.jl")
include("tools/tnmh_tools.jl")

mkpath("results")
set_seed(20240618)
betac = log(1 + sqrt(2)) / 2          # β_c = ln(1+√2)/2 ≈ 0.4407
log_finding(s) = open(io -> println(io, s), "results/FINDINGS.md", "a")
println("ready. β_c = ", round(betac, digits=6))


ready. β_c = 0.440687


## 0.1 — Deterministic `log q(config)`

`proposal_logprob(config, Lx, Ly, β, D)` runs the chain-rule sampler with every
spin **pinned** to `config`. Sanity: for a configuration the sampler itself drew,
the pinned `log q` must equal the `log q` the sampler reported (same arithmetic).


In [2]:
set_seed(1)
cfg, lq = sample_config_opt(3, 4, betac, 2)
lq_pin  = proposal_logprob(cfg, 3, 4, betac, 2)
@printf("sampler log q = %.12f\npinned  log q = %.12f\n|Δ| = %.2e\n", lq, lq_pin, abs(lq - lq_pin))
@assert abs(lq - lq_pin) < 1e-10
println("OK: pinned log q reproduces the sampler.")


sampler log q = -5.279393117281
pinned  log q = -5.279393117281
|Δ| = 0.00e+00
OK: pinned log q reproduces the sampler.


## 0.2 / 0.3 — Exact `π`, `Z` and the enumeration harness

`enumerate_weights(Lx,Ly,β,D)` iterates all `2^{Lx·Ly}` configs (building the
PEPS + bottom environments once), returns `logq, logπ, logw, E, w`, the worst-case
`C = max w`, `logZ`, and the built-in normalisation check `sum_q = Σ q`.


In [3]:
res = enumerate_weights(3, 4, betac, 2)
@printf("Σq = %.12f   Σπ = %.12f\nC(D=2) = %.8f   log Z = %.6f\n",
        res.sum_q, sum(exp.(res.logpi)), res.C, res.logZ)


Σq = 1.000000000000   Σπ = 1.000000000000
C(D=2) = 1.00128585   log Z = 10.126866


### Control A — proposal normalisation `Σ q = 1`

In [4]:
@assert abs(res.sum_q - 1) < 1e-8
println("Control A passed: Σq = ", round(res.sum_q, digits=12))


Control A passed: Σq = 1.0


### Control B — exact proposal ⇒ `C = 1`

For `Ly = 4` the horizontal cut needs bond `2^{⌈4/2⌉} = 4`, so the proposal is
exact at `D = 4` (and `C → 1`). At `D = 2` truncation is active ⇒ `C > 1`.


In [5]:
r_exact = nothing
println("D     C(3×4)")
for D in (2, 3, 4)
    global r_exact
    r = enumerate_weights(3, 4, betac, D)
    @printf("%-4d  %.10f\n", D, r.C)
    D == 4 && (r_exact = r)
end
@assert abs(r_exact.C - 1) < 1e-6
println("Control B passed: C → 1 at the exact bond dimension.")

D     C(3×4)
2     1.0012858461
3     1.0001425733
4     1.0000000000
Control B passed: C → 1 at the exact bond dimension.


## Cross-check vs the Python implementation (plan operating principles)

The two independent code paths (Julia ITensors vs Python numpy) must agree on
`log q` to ~1e-8. We write `log q` for a fixed set of configs to a CSV and, if
the Python file exists, compare. **Run `py/phase0b_python_crosscheck.ipynb` first**
to produce `results/crosscheck_py.csv`. `config_from_int` uses column-major bit
order — the Python notebook replicates it so the configs match exactly.


In [6]:
ids = collect(0:31)
open("results/crosscheck_jl.csv", "w") do io
    println(io, "id,logq")
    for id in ids
        @printf(io, "%d,%.15e\n", id, proposal_logprob(config_from_int(id, 3, 4), 3, 4, betac, 2))
    end
end
println("wrote results/crosscheck_jl.csv")

if isfile("results/crosscheck_py.csv")
    py = readdlm("results/crosscheck_py.csv", ',', skipstart=1)
    maxd = 0.0
    for row in 1:size(py, 1)
        id = Int(py[row, 1])
        lqjl = proposal_logprob(config_from_int(id, 3, 4), 3, 4, betac, 2)
        maxd = max(maxd, abs(lqjl - py[row, 2]))
    end
    @printf("Julia vs Python  max|Δ log q| over %d configs = %.2e\n", size(py, 1), maxd)
    @assert maxd < 1e-8
    println("Cross-check PASSED (≤ 1e-8).")
else
    println("results/crosscheck_py.csv not found — run the Python notebook, then re-run this cell.")
end


wrote results/crosscheck_jl.csv
Julia vs Python  max|Δ log q| over 32 configs = 2.66e-15
Cross-check PASSED (≤ 1e-8).


## Save + record finding

In [7]:
jldsave("results/phase0_infrastructure.jld2";
        logq=res.logq, logpi=res.logpi, logw=res.logw, E=res.E,
        C=res.C, logZ=res.logZ, Lx=3, Ly=4, D=2, beta=betac)
log_finding("\n## Phase 0 — Infrastructure")
log_finding("- Control A: Σq = $(round(res.sum_q, digits=12)) (=1).")
log_finding("- Control B: C(3×4, D=4 exact) = $(round(r_exact.C, digits=8)) (→1); C(D=2)=$(round(res.C,digits=6)).")
log_finding("- Pinned log q matches the sampler to <1e-10; Julia↔Python agree ≤1e-8 (see crosscheck CSVs).")
println("saved results/phase0_infrastructure.jld2 and appended FINDINGS.md")


saved results/phase0_infrastructure.jld2 and appended FINDINGS.md
